# Benders Decomposition with Plasmo.jl

## Summary

This notebook decomposes a linopy capacity-expansion model into a Benders master problem
plus one subproblem per planning year, using `linopy.contrib.plasmo` -- an experimental
bridge to [Plasmo.jl](https://github.com/plasmo-dev/Plasmo.jl) and
[PlasmoBenders.jl](https://github.com/plasmo-dev/PlasmoBenders.jl). It reimplements the
Python side of [Linopy2Plasmo.jl](https://github.com/leonardgoeke/Linopy2Plasmo.jl), driving
everything from Python instead of the Julia REPL.

See [Benders Decomposition with Plasmo.jl](plasmo-benders.rst) in the user guide for the full
conceptual explanation (partitions, topologies, algorithms). This notebook is the worked
example: build a partition, decompose, solve with Benders, and cross-check the result
against a monolithic solve of the same model.

**Requires a Julia installation** managed transparently by `juliacall`/`pyjuliapkg` -- see
the *Installing the Julia side* section of the linked page. The first run resolves and
precompiles the Julia packages, which can take a few minutes.

## The model

`testProblem.nc` is an energy-system capacity-expansion problem (ZEN-garden-style): it
decides technology `capacity` and `capacity_addition` per node and year, subject to energy
balance, carbon-budget, and technology constraints, minimizing `net_present_cost`. It has 3
planning years (`set_time_steps_yearly`) and several nodes (`set_location`) -- exactly the
two dimensions the decomposition below splits along.

In [ ]:
import numpy as np

import linopy
from linopy.contrib.plasmo import Partition, PlasmoModel, benders, group, has, optimize

## Load and clean the model

`testProblem.nc` stores a few constraint-term cells with a valid variable reference but a
`NaN` coefficient -- a padding artifact of its ragged term axis. We zero those coefficients,
then let `sanitize_zeros` drop the now-empty terms so the exported matrix has no `NaN` and no
empty rows.

In [ ]:
def clean_model(m: linopy.Model) -> None:
    for name in m.constraints:
        con = m.constraints[name]
        con._update_data(coeffs=con.coeffs.where(~np.isnan(con.coeffs), 0.0))
    m.constraints.sanitize_zeros()


m = linopy.read_netcdf("testProblem.nc")
clean_model(m)
m

The model has 37 variables and 50 constraints (as linopy sees them -- each one an N-D array
of scalar variables/constraints; the flattened LP has tens of thousands of rows and columns,
see below). `capacity` is a representative decision variable, dimensioned over
technology/node/year:

In [ ]:
m.variables["capacity"].dims, m.variables["capacity"].shape

## Defining the partition

A [`Partition`](plasmo-benders.rst) assigns every constraint to a decomposition
node. Here we want: constraints that don't vary by year or by node go to a `top` master
(cross-year coupling, like the carbon budget); everything else splits into one `sub` per
year. This mirrors Linopy2Plasmo.jl's example `split_tup`/`def_dic`.

In [ ]:
partition = Partition(
    top=~has("set_time_steps_yearly") | ~has("set_nodes"),
    sub=group("set_time_steps_yearly") & has("set_nodes"),
)

- `top`: constraints with **no** year dimension, or **no** node dimension -- e.g. the overall
  carbon budget, which sums across all years and nodes.
- `sub`: constraints that have **both** a year and a node dimension, split into one node per
  distinct year value (`group("set_time_steps_yearly")`) -- e.g. the nodal energy balance,
  evaluated separately per year.

`~has(...)` negates a scalar predicate; `|` combines two scalars (allowed); `&` gates the
scattering `group(...)` predicate with `has(...)` so only node-and-year-dimensioned rows are
split by year. Declaration order matters: `top` is declared first, so it becomes the Benders
master.

## Building the graph

`PlasmoModel` only *plans* the partition on construction (cheap, pure Python/numpy); the
Julia `OptiGraph` is built lazily, streaming one node's LP block at a time so peak memory is
one block, not the whole decomposed problem.

In [ ]:
pm = PlasmoModel(m, partition)
pm

Four nodes came out: the `top` master plus `sub[0]`, `sub[1]`, `sub[2]` (one per year), linked
by 60 equality constraints on the variables shared between the master and each year's
subproblem (e.g. installed capacity carried over from one year to the next).

In [ ]:
%time pm.graph

## Interacting through Julia underneath

In [ ]:
from juliacall import Main as jl

jl.seval("using Plasmo")

In [ ]:
jl.all_subgraphs(pm.graph)

## Solving with Benders

`benders()` requires a flat topology (master and subproblems as siblings under the root --
the default), builds the graph on first use, and runs `PlasmoBenders.jl`'s
`BendersAlgorithm` to convergence.

In [ ]:
benders(pm, warm_start=False, regularize=True)
obj_benders = pm.result().solution.objective
print(f"Benders objective: {obj_benders:.6g}")

In [ ]:
pm.result()

In [ ]:
m.assign_result(pm.result())

## Cross-checking against a monolithic solve

As a correctness check, solve the same (undecomposed) model directly with HiGHS and compare
objectives.

*(This particular model triggers an unrelated linopy bug when writing constraint duals back
onto a ragged dimension -- the objective is already assigned by the time that happens, so we
tolerate the exception here; it is independent of the Benders decomposition above.)*

In [ ]:
m2 = linopy.read_netcdf("testProblem.nc")
clean_model(m2)
try:
    m2.solve(solver_name="highs")
except Exception as e:  # noqa: BLE001
    print(f"(tolerated linopy dual-assignment bug: {e})")
obj_monolithic = m2.objective.value

rel = abs(obj_benders - obj_monolithic) / max(1.0, abs(obj_monolithic))
print(f"Benders objective:    {obj_benders:.6g}")
print(f"Monolithic objective: {obj_monolithic:.6g}")
print(f"relative difference:  {rel:.2e}")
assert rel < 1e-4, "Benders objective does not match the monolithic solve"

The objectives match to numerical tolerance -- the decomposed solve reproduces the
undecomposed one, as expected for a converged Benders run on a continuous LP.

## Reading back the solution

`assign_to_model()` writes the decomposed solution back onto `m` via linopy's normal
`assign_result` path, so `m.solution` and `variable.solution` work as they would after
`m.solve()`. Constraint duals are left empty (Benders subproblems don't yield duals for the
original, undecomposed model).

In [ ]:
pm.assign_to_model()
m.solution["capacity"]

## Monolithic solve as an alternative algorithm

The same `PlasmoModel` graph also supports `optimize()`, which solves the whole graph at
once (every subgraph, links enforced as hard constraints) rather than via Benders iteration.
Unlike `benders()`, it works with any topology, not just a flat one -- useful as a quick
correctness check on the graph construction itself, independent of the Benders algorithm.

In [ ]:
pm2 = PlasmoModel(m, partition)
optimize(pm2)
pm2.result().solution.objective

In [ ]:
m.solve(io_api="direct")

## Limitations

- Continuous variables only (inherited from PlasmoBenders.jl).
- One node per `Partition` cell -- no separate, finer variable-side partition yet.
- This experimental module lives in `linopy.contrib.plasmo` and is not covered by CI.

## References

- [Plasmo.jl documentation](https://plasmo-dev.github.io/Plasmo.jl/stable/)
- [PlasmoBenders.jl](https://github.com/plasmo-dev/PlasmoBenders.jl)
- [Linopy2Plasmo.jl](https://github.com/leonardgoeke/Linopy2Plasmo.jl) -- the original
  Julia-only prototype this module reimplements the Python side of.